In [1]:
from typing import TypedDict, Literal

class State(TypedDict):
    value: int
    decision: Literal["even", "odd"]  # 조건 판단 결과 저장

In [2]:
# 조건을 판단하고 판단 결과를 State에 저장하는 Node
def check_condition_node(state: State):
    print("check_condition_node 실행")
    print(f"value = {state['value']}")

    # Node에서 조건을 판단하고 State에 저장
    if state["value"] % 2 == 0:
        decision = "even"
        print("판단: 짝수")
    else:
        decision = "odd"
        print("판단: 홀수")

    print("=" * 20)

    # 판단 결과를 State에 저장하여 반환
    return {"decision": decision}

# 짝수일 때 실행되는 Node
def even_node(state: State):
    print("even_node 실행: 짝수입니다!")
    print(f"value = {state['value']}")
    return state

# 홀수일 때 실행되는 Node
def odd_node(state: State):
    print("odd_node 실행: 홀수입니다!")
    print(f"value = {state['value']}")
    return state


In [3]:
def router(state: State) -> Literal["even", "odd"]:
    print("router 실행")
    print(f"State에 저장된 decision: {state['decision']}")

    # State에 이미 저장된 판단 결과를 단순히 반환만 함
    if state["decision"] == "even":
        print("짝수 Node로 이동")
        return "even"
    else:
        print("홀수 Node로 이동")
        return "odd"        

In [4]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(State)

graph_builder.add_node("check_condition", check_condition_node)
graph_builder.add_node("even", even_node)
graph_builder.add_node("odd", odd_node)

graph_builder.add_edge(START, "check_condition")
graph_builder.add_conditional_edges(
    "check_condition",   
    router,         # router로 다음 Node 결정
    {
        "even": "even",
        "odd": "odd",
    }
)

graph_builder.add_edge("even", END)
graph_builder.add_edge("odd", END)

graph = graph_builder.compile()



In [6]:
initial_message = {"value": 2}

result = graph.invoke(initial_message)
print("최종 결과:", result)
print('-' * 30)
### 홀수 입력하기
initial_message = {"value": 3}
result = graph.invoke(initial_message)
print("최종 결과:", result)


check_condition_node 실행
value = 2
판단: 짝수
router 실행
State에 저장된 decision: even
짝수 Node로 이동
even_node 실행: 짝수입니다!
value = 2
최종 결과: {'value': 2, 'decision': 'even'}
------------------------------
check_condition_node 실행
value = 3
판단: 홀수
router 실행
State에 저장된 decision: odd
홀수 Node로 이동
odd_node 실행: 홀수입니다!
value = 3
최종 결과: {'value': 3, 'decision': 'odd'}
